> ## SOLUTION / ANSWER KEY &mdash; Lab 8.6
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-06-handoff.ipynb`](../lab-06-handoff.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.6 &mdash; Explicit Handoff (Capped)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Let each agent return the next agent, or 'done'
- Walk the handoff path from a starting agent
- Cap total handoffs so two agents can't loop forever

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Message passing & shared state](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-06"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Agents coordinate by **explicit handoffs**: an agent signals *&ldquo;done&rdquo;* or *&ldquo;this needs the
tech agent&rdquo;*, and control passes on. The danger is a **handoff loop** &mdash; A hands to B, B hands
back to A, forever (deck slide 19) &mdash; so you **cap total handoffs**, exactly like `max_iterations`.
Explicit handoffs plus a cap are what keep a team from becoming a mob.

In [ ]:
# Each agent is a function returning the NEXT agent's name, or "done".
print("handoff: supervisor -> billing -> tech -> done")

## Build it
Complete `run_handoffs`: stop at `"done"`, otherwise follow the handoff, all under a cap.

In [ ]:
def run_handoffs(start, agents, max_handoffs=5):
    current, path = start, []
    for _ in range(max_handoffs):
        path.append(current)
        nxt = agents[current]()
        if nxt == "done":
            break
        current = nxt
    return path

AGENTS = {"supervisor": lambda: "billing", "billing": lambda: "tech", "tech": lambda: "done"}
LOOP   = {"a": lambda: "b", "b": lambda: "a"}   # two agents that would loop forever

try:
    print("path :", run_handoffs("supervisor", AGENTS))
    print("loop :", run_handoffs("a", LOOP, max_handoffs=4))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The handoff **path** is walked in order and stops at `"done"` &mdash; explicit control passing, not magic.
- Point it at `LOOP` (two agents that hand back and forth) and the **cap** stops it dead &mdash; the multi-agent `max_iterations`.
- A lower cap stops sooner: the cap is your dial between *let the team finish* and *never run away*.

## Your turn (open task &mdash; no grader)
Add an agent that hands off **conditionally** (returns `"tech"` sometimes and `"done"` other times), and walk
a few runs. **What good looks like:** normal runs terminate at `"done"` well under the cap, while a
deliberately looping set is always stopped by the cap &mdash; no polite pair of agents can hand back and forth
forever.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
import itertools
flip = itertools.cycle(["tech", "done"])                 # sometimes hands to tech, sometimes finishes
COND = {"supervisor": lambda: "billing", "billing": lambda: next(flip), "tech": lambda: "done"}
print("run 1 :", run_handoffs("supervisor", COND))       # terminates at "done", under the cap
print("run 2 :", run_handoffs("supervisor", COND))
print("loop  :", run_handoffs("a", LOOP, max_handoffs=4)) # a looping pair is always stopped by the cap

---
### Key takeaway
Explicit handoffs plus a cap are the multi-agent version of the agent loop with max_iterations. Without the cap, two polite agents hand back and forth forever.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>